[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1AJd1EW5Y9ZUn2DABUdUFHkYQTOBGsyW4/view?usp=drive_link)

# Prompt Evaluation – Basic Prompt Comparison

This notebook demonstrates how to compare different system prompts on the same questions. Floeval runs each question with each prompt, generates responses, and scores them so you can see which instruction performs better.

**Objectives**
- Install Floeval and configure credentials
- Create a prompts file with named prompt variants
- Build a partial dataset with `prompt_ids`
- Run evaluation and compare scores across prompts

## 3. Create the Prompts File

A YAML file is created with three prompt variants (IDs 1, 2, 3). Each prompt includes a `template` field containing the system instruction.

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
from pathlib import Path

prompts_content = """
prompts:
  "1":
    template: "Answer the question directly and concisely."
  "2":
    template: "Answer the question with a brief explanation of your reasoning."
  "3":
    template: "Answer the question in a structured format with numbered points."
"""
Path("prompts.yaml").write_text(prompts_content.strip())
print("Created prompts.yaml")

## 2. Imports

The following cell imports the evaluation components and the LLM configuration schema.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 5. Configure the LLM

The LLM configuration is built using environment variables. Set `OPENAI_API_KEY` in your environment or replace the placeholder.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 4. Load the Dataset

The dataset is loaded with `user_input` and `prompt_ids` for each sample. The `partial_dataset=True` flag indicates that Floeval will generate responses at runtime.

In [ ]:
dataset = DatasetLoader.from_samples(
    [
        {"user_input": "What is the capital of France?", "prompt_ids": ["1", "2", "3"]},
        {"user_input": "What is RAG in machine learning?", "prompt_ids": ["1", "2", "3"]},
    ],
    partial_dataset=True,
)
print(f"Dataset loaded: {len(dataset.samples)} samples")

## 7. Create and Run the Evaluation

The `prompts_file` and `dataset_generator_model` parameters are passed to the evaluation. Floeval generates a response for each (sample, prompt_id) pair and scores them.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file="prompts.yaml",
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

## 8. Compare Scores by Prompt

Each result row includes `prompt_id`. Results are grouped by prompt to compare average scores across instructions.

In [ ]:
from collections import defaultdict

by_prompt = defaultdict(list)
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    for key, data in sr.get("metrics", {}).items():
        if data.get("score") is not None:
            by_prompt[pid].append(data["score"])

for pid, scores in by_prompt.items():
    avg = sum(scores) / len(scores) if scores else 0
    print(f"{pid}: avg score = {avg:.4f} (n={len(scores)})")

## Summary

This notebook demonstrated how to compare different system prompts on the same questions using Floeval.

The key components included:

1. **Prompts File**: A YAML file with three prompt variants (IDs `1`, `2`, `3`) was created.
2. **Dataset with Prompt IDs**: A partial dataset was loaded with `prompt_ids` ["1", "2", "3"] for each sample.
3. **Evaluation Execution**: The evaluation was run with `prompts_file` and `dataset_generator_model` to generate and score responses for each (sample, prompt_id) pair.
4. **Score Comparison**: Results were grouped by `prompt_id` to compare average scores across instructions.

This example showcases the workflow for evaluating and comparing prompt performance across questions.